### Dataset MNIST com CNN

Para este experimento foi utilizada a base de dados original do MNIST, obtida por download direto dos arquivos IDX compactados. Diferentemente da utilização do carregador disponível em bibliotecas como Keras, os arquivos foram lidos manualmente utilizando Python, realizando a extração das imagens e rótulos para memória. Em seguida, foi aplicado o pré-processamento necessário para a rede neural convolucional, incluindo a normalização dos valores dos pixels para o intervalo entre 0 e 1 e a reorganização das imagens para o formato esperado pelas camadas convolucionais do PyTorch.

A classificação foi realizada utilizando uma Rede Neural Convolucional (CNN) composta por três camadas convolucionais e camadas de pooling, seguida de uma camada totalmente conectada para a classificação dos dígitos. O modelo apresentou uma acurácia próxima de 98% no conjunto de teste, demonstrando excelente capacidade de reconhecimento dos dígitos manuscritos. Esse resultado é compatível com o esperado para arquiteturas convolucionais aplicadas ao MNIST, evidenciando que a rede foi capaz de extrair características relevantes das imagens e generalizar adequadamente para exemplos não utilizados durante o treinamento.

In [9]:
import os
import urllib.request

BASE_URL = "https://storage.googleapis.com/cvdf-datasets/mnist/"

FILES = [
    "train-images-idx3-ubyte.gz",
    "train-labels-idx1-ubyte.gz",
    "t10k-images-idx3-ubyte.gz",
    "t10k-labels-idx1-ubyte.gz"
]

os.makedirs("mnist_raw", exist_ok=True)

for file in FILES:
    path = os.path.join("mnist_raw", file)

    if not os.path.exists(path):
        print(f"Downloading {file}")
        urllib.request.urlretrieve(BASE_URL + file, path)

In [10]:
import gzip
import struct
import numpy as np

def load_images(path):
    with gzip.open(path, "rb") as f:
        magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
        images = np.frombuffer(
            f.read(),
            dtype=np.uint8
        )
        images = images.reshape(num, rows, cols)

    return images

def load_labels(path):
    with gzip.open(path, "rb") as f:
        magic, num = struct.unpack(">II", f.read(8))
        labels = np.frombuffer(
            f.read(),
            dtype=np.uint8
        )

    return labels

In [11]:
train_images = load_images(
    "mnist_raw/train-images-idx3-ubyte.gz"
)

train_labels = load_labels(
    "mnist_raw/train-labels-idx1-ubyte.gz"
)

test_images = load_images(
    "mnist_raw/t10k-images-idx3-ubyte.gz"
)

test_labels = load_labels(
    "mnist_raw/t10k-labels-idx1-ubyte.gz"
)

print(train_images.shape)
print(train_labels.shape)

print(test_images.shape)
print(test_labels.shape)

(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


### Implementação de CNN 2D retirada de https://github.com/fboldt/aulasann/blob/main/aula06d%20-%20mnist%20conv%20torch.ipynb

In [12]:
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [13]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score

class TorchCNN2D(nn.Module):
  def __init__(self, input_shape, num_classes):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels=input_shape[0],
                           out_channels=32, kernel_size=3)
    self.pool1 = nn.MaxPool2d(kernel_size=2)
    self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
    self.pool2 = nn.MaxPool2d(kernel_size=2)
    self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)
    self.flatten = nn.Flatten()
    self.fc = nn.Linear(in_features=1152, out_features=num_classes)
  def forward(self, x):
    x = F.relu(self.conv1(x))
    x = self.pool1(x)
    x = F.relu(self.conv2(x))
    x = self.pool2(x)
    x = F.relu(self.conv3(x))
    x = self.flatten(x)
    x = self.fc(x)
    x = F.softmax(x, dim=1)
    return x

input_shape = (1, 28, 28)
num_classes = 10
model = TorchCNN2D(input_shape, num_classes)

In [14]:
from sklearn.base import BaseEstimator, TransformerMixin

class TorchWrappedNN(BaseEstimator, ClassifierMixin):
  def __init__(self, epochs=5, batch_size=128, model_fabric=TorchCNN2D):
    self.epochs = epochs
    self.batch_size = batch_size
    self.model_fabric = model_fabric

  def fit(self, X, y):
    self.labels, ids = torch.unique(torch.tensor(y), return_inverse=True)
    self.model = self.model_fabric(input_shape=X.shape[1:], num_classes=len(self.labels)).to(device)
    self.optimizer = optim.RMSprop(self.model.parameters(), lr=0.001)
    self.loss_fn = nn.CrossEntropyLoss()
    train_dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(ids, dtype=torch.long))
    train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
    for epoch in range(self.epochs):
      for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        self.optimizer.zero_grad()
        y_pred = self.model(batch_X)
        loss = self.loss_fn(y_pred, batch_y)
        loss.backward()
        self.optimizer.step()
    return self

  def predict(self, X):
    X = torch.tensor(X, dtype=torch.float32).to(device)
    y_pred = self.model(X)
    return self.labels[torch.argmax(y_pred, dim=1).cpu().numpy()]

In [15]:
from sklearn.pipeline import Pipeline

class Divide255(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X):
    return X / 255.0

class Shape2Torch(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X):
    return X.reshape((-1, 1, 28, 28))

pipeline = Pipeline([
    ("scaler", Divide255()),
    ("shape", Shape2Torch()),
    ("model", TorchWrappedNN(epochs=10, model_fabric=TorchCNN2D))
])

pipeline.fit(train_images, train_labels)
y_pred = pipeline.predict(test_images)
accuracy_score(test_labels, y_pred)

/tmp/ipykernel_5648/1016415351.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(ids, dtype=torch.long))
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


0.989